# 09 – Evaluation: Métricas de Evaluación

**Proyecto:** Análisis y predicción del subempleo por insuficiencia de horas en el Perú – EPEN 2024  
**Target:** `target_subempleo_horas` (1 = subempleado por horas · 0 = no subempleado)  
**Objetivo:** Calcular y analizar las métricas de desempeño del modelo seleccionado sobre el conjunto de prueba real.

> **Prerequisito:** Ejecuta primero `06_feature_selection/selected_variables.ipynb` y `07_modelling/01_baseline_model.ipynb`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve,
    classification_report,
)
import joblib
from pathlib import Path

# ── Rutas ──────────────────────────────────────────────────────────────────────
SEL_DIR     = Path('../data/selected')
MODEL_DIR   = Path('../models')
RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET      = 'target_subempleo_horas'
CLASS_NAMES = ['No subempleado por horas', 'Subempleado por horas']

# ── Carga de datos reales (sin fallback sintético) ─────────────────────────────
required = {
    'X_train': SEL_DIR / 'X_train_selected.csv',
    'X_test':  SEL_DIR / 'X_test_selected.csv',
    'y_train': SEL_DIR / 'y_train_selected.csv',
    'y_test':  SEL_DIR / 'y_test_selected.csv',
}
for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Archivo requerido no encontrado: {path}\n"
            "Ejecuta primero: 06_feature_selection/selected_variables.ipynb"
        )

X_train = pd.read_csv(required['X_train'])
X_test  = pd.read_csv(required['X_test'])

def load_target(path, name):
    df = pd.read_csv(path)
    if TARGET in df.columns:
        return df[TARGET].reset_index(drop=True)
    if df.shape[1] == 1:
        return df.iloc[:, 0].reset_index(drop=True)
    raise ValueError(f"No se encontró '{TARGET}' en {name}")

y_train = load_target(required['y_train'], 'y_train')
y_test  = load_target(required['y_test'],  'y_test')

print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"y_train – clase 1 (subempleado): {y_train.mean():.2%}")
print(f"y_test  – clase 1 (subempleado): {y_test.mean():.2%}")

# ── Carga del modelo ganador ───────────────────────────────────────────────────
# Ganador según model_comparison.csv: Logistic Regression (class_weight='balanced')
# Criterio: mayor F1-score clase 1 (0.475) y ROC-AUC (0.697) en conjunto de prueba.
model_path = MODEL_DIR / 'logistic_regression_balanced.pkl'
if not model_path.exists():
    raise FileNotFoundError(
        f"Modelo no encontrado: {model_path}\n"
        "Ejecuta primero: 07_modelling/01_baseline_model.ipynb"
    )

model  = joblib.load(model_path)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(f'\nModelo cargado : {model_path.name}')
print('Predicciones generadas correctamente.')


## 1. Métricas principales

In [ ]:
metrics = {
    'Accuracy':                   accuracy_score(y_test, y_pred),
    'Precision (clase 1)':        precision_score(y_test, y_pred, zero_division=0),
    'Recall (clase 1)':           recall_score(y_test, y_pred, zero_division=0),
    'F1-Score (clase 1)':         f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC':                    roc_auc_score(y_test, y_prob),
    'Average Precision (PR-AUC)': average_precision_score(y_test, y_prob),
}

print('=== Métricas de Evaluación – Modelo Ganador ===')
print(f'    Logistic Regression (class_weight="balanced")\n')
for k, v in metrics.items():
    print(f'  {k:35s}: {v:.4f}')

print('\n=== Reporte completo ===')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 2. Curva ROC y Curva Precisión-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
axes[0].fill_between(fpr, tpr, alpha=0.15, color='steelblue')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Curva Precisión-Recall
prec, rec, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
axes[1].step(rec, prec, color='darkorange', lw=2, where='post', label=f'AP = {ap:.3f}')
axes[1].fill_between(rec, prec, alpha=0.15, color='darkorange', step='post')
axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', lw=1, label='Baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precisión-Recall')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Análisis del umbral de clasificación

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    threshold_results.append({
        'Threshold': t,
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall': recall_score(y_test, y_pred_t, zero_division=0),
        'F1': f1_score(y_test, y_pred_t, zero_division=0),
    })

thresh_df = pd.DataFrame(threshold_results)
thresh_df.set_index('Threshold')[['Precision', 'Recall', 'F1']].plot(
    figsize=(10, 4), marker='o'
)
plt.title('Métricas vs. Umbral de clasificación')
plt.ylabel('Score')
plt.tight_layout()
plt.show()

best_t = thresh_df.loc[thresh_df['F1'].idxmax()]
print(f'Umbral óptimo (F1 máximo): {best_t["Threshold"]:.2f} → F1={best_t["F1"]:.3f}')

In [ ]:
# Guardar métricas en data/results/final_metrics.csv
pd.DataFrame([metrics]).to_csv(RESULTS_DIR / 'final_metrics.csv', index=False)
print(f'Métricas guardadas: {RESULTS_DIR / "final_metrics.csv"}')
